In [ ]:
# Cell 1: Install system dependencies
!apt-get install -y bedtools samtools

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Download hg38 reference genome and index (skipped if exists)
%%bash
mkdir -p /content/drive/MyDrive/ML_Project/data
cd /content/drive/MyDrive/ML_Project/data

if [ ! -f hg38.fa ]; then
    echo "Downloading hg38.fa.gz ..."
    wget -q http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    echo "Decompressing ..."
    gunzip -k hg38.fa.gz
fi

if [ ! -f hg38.fa.fai ]; then
    echo "Indexing ..."
    samtools faidx hg38.fa
fi

In [ ]:
# Cell 4: Parse narrowPeak, center 101bp window on summit, filter boundaries
import pandas as pd

RAW = '/content/drive/MyDrive/ML_Project/data/sp1_raw_data.narrowPeak.bed'
CENTERED = '/content/drive/MyDrive/ML_Project/data/sp1_centered_101bp.bed'

COLS = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal', 'pval', 'qval', 'peak']

df = pd.read_csv(RAW, sep='\t', header=None, names=COLS)
df[['start', 'end', 'peak']] = df[['start', 'end', 'peak']].astype(int)

df['summit'] = df['start'] + df['peak']
df['new_start'] = df['summit'] - 50
df['new_end'] = df['summit'] + 51

initial = len(df)
df = df[(df['new_start'] >= 0) & (df['new_end'] > df['new_start'])]
print(f"Filtered: {initial} -> {len(df)} peaks")

df[['chrom', 'new_start', 'new_end']].to_csv(CENTERED, sep='\t', header=False, index=False)
print(f"Saved: {CENTERED}")

In [ ]:
# Cell 5: Extract chromosome sizes from FASTA index
%%bash
cd /content/drive/MyDrive/ML_Project/data
cut -f 1,2 hg38.fa.fai > hg38.chrom.sizes
echo "Generated hg38.chrom.sizes"

In [ ]:
# Cell 6: Extract positive FASTA, generate matched negative set via bedtools shuffle
%%bash
cd /content/drive/MyDrive/ML_Project/data

echo "Extracting positive sequences ..."
bedtools getfasta -fi hg38.fa -bed sp1_centered_101bp.bed -fo sp1_positive_101bp.fasta

echo "Generating negative coordinates ..."
bedtools shuffle \
    -i sp1_centered_101bp.bed \
    -g hg38.chrom.sizes \
    -excl sp1_centered_101bp.bed \
    -noOverlapping \
    -seed 42 \
    > sp1_negative_101bp.bed

echo "Extracting negative sequences ..."
bedtools getfasta -fi hg38.fa -bed sp1_negative_101bp.bed -fo sp1_negative_101bp.fasta

POS=$(grep -c "^>" sp1_positive_101bp.fasta)
NEG=$(grep -c "^>" sp1_negative_101bp.fasta)
echo "Positive: $POS  Negative: $NEG"

In [ ]:
# Cell 7: Filter sequences with N, standardize to uppercase
import os

def clean_fasta(input_path, output_path):
    total, valid, discarded = 0, 0, 0
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        header, seq_parts = None, []
        for line in fin:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    total += 1
                    seq = ''.join(seq_parts).upper()
                    if 'N' not in seq:
                        fout.write(f"{header}\n{seq}\n")
                        valid += 1
                    else:
                        discarded += 1
                header, seq_parts = line, []
            else:
                seq_parts.append(line)
        if header is not None:
            total += 1
            seq = ''.join(seq_parts).upper()
            if 'N' not in seq:
                fout.write(f"{header}\n{seq}\n")
                valid += 1
            else:
                discarded += 1
    print(f"{os.path.basename(input_path)}: {valid}/{total} retained ({discarded} discarded)")
    return total, valid, discarded

BASE = "/content/drive/MyDrive/ML_Project/data"
t1, v1, d1 = clean_fasta(f"{BASE}/sp1_positive_101bp.fasta", f"{BASE}/sp1_positive_101bp_clean.fasta")
t2, v2, d2 = clean_fasta(f"{BASE}/sp1_negative_101bp.fasta", f"{BASE}/sp1_negative_101bp_clean.fasta")

print(f"\nFinal: pos={v1}, neg={v2}")
if v1 == v2:
    print("Classes balanced.")
else:
    print(f"Class imbalance: {abs(v1-v2)} sequences difference.")